# Notebook 1: Food Recognition Foundations (Real-World CNN From Scratch)
### Minor: AI for Society | Personal Project: Food Recognition & Calorie Estimator
**Author:** Storm Burg  
**Approach:** 100% Custom Convolutional Neural Network (CNN) trained from scratch  
**Datasets:** Real-World Fruit & Vegetable Dataset (*Kritik Seth*) + *Fruits-360*  
**Objective:** Train our own custom deep learning model capable of recognizing fruits and vegetables **in real-world environments** (on tables, in kitchens, under natural lighting) without relying on pre-trained networks.

---
## Theoretical Context & Project Roadmap
1. **The Domain Gap Challenge:**
   - Synthetic studio datasets (like *Fruits-360*) isolate fruit against uniform white backgrounds. Models trained exclusively on studio data often fail when deployed to mobile cameras because they memorize the white background as a shortcut.
2. **Solving the Problem at the Data Level (No Pretrained Transfer Learning):**
   - Instead of using pre-trained models like MobileNet or ResNet, we remain true to training our **own model architecture from scratch**.
   - We solve the real-world recognition issue by integrating **real-world in-the-wild photography** (fruits on wooden tables, plates, cutting boards, and store displays).
3. **Learning Outcomes:**
   - Understand image tensor representations and batch pipelines.
   - Design a capable custom CNN with batch normalization, dropout, and spatial downsampling.
   - Master data augmentation to bridge domain gaps.
   - Deploy and test the custom model live with an interactive smartphone scanner via Gradio.


## 1. Setup & Environment
We verify GPU availability (an NVIDIA T4 or similar in Google Colab), set random seeds for reproducibility, and install/import required libraries.


In [ ]:
# Install helper libraries
!pip install -q kagglehub torch torchvision matplotlib seaborn scikit-learn tqdm gradio pillow

import os
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import datasets, transforms

# Set reproducible seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")


## 2. Mount Google Drive for Checkpoint Persistence
Colab instances are ephemeral. We mount Google Drive to persist our trained model checkpoints under `/content/drive/MyDrive/Minor AI/models/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/Minor AI/models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoint directory ready: {SAVE_DIR}")


## 3. Dataset Acquisition: Real-World Fruit & Vegetable Images
We download the **Fruit and Vegetable Image Recognition** dataset (`kritikseth/fruit-and-vegetable-image-recognition`) via `kagglehub`.  
Unlike studio turntable datasets, these images were captured in real-world environments: kitchens, wooden tables, market shelves, and plates.

We select a core set of 10 popular fruits and vegetables:
- **Apple**
- **Banana**
- **Lemon**
- **Orange**
- **Strawberry**
- **Tomato**
- **Corn**
- **Carrot**
- **Cucumber**
- **Bell Pepper**


In [ ]:
import kagglehub

print("Downloading Real-World Fruits & Vegetables dataset via kagglehub...")
real_dataset_path = kagglehub.dataset_download("kritikseth/fruit-and-vegetable-image-recognition")
print(f"Dataset path: {real_dataset_path}")

raw_train_dir = os.path.join(real_dataset_path, "train")
raw_test_dir = os.path.join(real_dataset_path, "validation")
if not os.path.exists(raw_test_dir):
    raw_test_dir = os.path.join(real_dataset_path, "test")

# Target class selection
TARGET_CLASSES = [
    "apple",
    "banana",
    "lemon",
    "orange",
    "strawberry",
    "tomato",
    "corn",
    "carrot",
    "cucumber",
    "bell pepper"
]

LOCAL_DATA_DIR = "/content/real_dataset_subset"
train_dir = os.path.join(LOCAL_DATA_DIR, "train")
test_dir = os.path.join(LOCAL_DATA_DIR, "test")

if os.path.exists(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

def copy_selected_classes(src_root, dst_root, class_list):
    available = os.listdir(src_root)
    for target in class_list:
        matched = None
        for folder in available:
            if folder.lower() == target.lower():
                matched = folder
                break
        if not matched:
            for folder in available:
                if target.lower() in folder.lower():
                    matched = folder
                    break
        if matched:
            src_folder = os.path.join(src_root, matched)
            dst_folder = os.path.join(dst_root, target.replace(" ", "_"))
            shutil.copytree(src_folder, dst_folder)
            print(f"  Copied {matched} -> {dst_folder}")
        else:
            print(f"  [WARNING] Could not find folder for: {target}")

print("\nPreparing training split...")
copy_selected_classes(raw_train_dir, train_dir, TARGET_CLASSES)

print("\nPreparing test split...")
copy_selected_classes(raw_test_dir, test_dir, TARGET_CLASSES)

active_classes = sorted(os.listdir(train_dir))
print(f"\nActive Classes ({len(active_classes)}): {active_classes}")


## 4. Exploratory Data Analysis (EDA)
Let's inspect our real-world dataset:
1. **Class distribution**: Number of images per fruit category.
2. **Visual inspection**: Notice that these images contain real kitchen counters, wooden boards, and variable lighting.


In [ ]:
# 1. Sample distribution
train_counts = {cls: len(os.listdir(os.path.join(train_dir, cls))) for cls in active_classes}
test_counts = {cls: len(os.listdir(os.path.join(test_dir, cls))) for cls in active_classes}

plt.figure(figsize=(12, 4))
x = np.arange(len(active_classes))
width = 0.35

plt.bar(x - width/2, [train_counts[c] for c in active_classes], width, label='Train', color='#3b82f6')
plt.bar(x + width/2, [test_counts[c] for c in active_classes], width, label='Test / Val', color='#10b981')
plt.xticks(x, [c.replace('_', ' ').title() for c in active_classes], rotation=45, ha='right')
plt.title("Sample Count per Class (Real-World Dataset)")
plt.ylabel("Number of Images")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Total training samples: {sum(train_counts.values())}")
print(f"Total test/validation samples: {sum(test_counts.values())}")


In [ ]:
# 2. Visual inspection of real-world sample images
plt.figure(figsize=(14, 6))
cols = 5
rows = int(np.ceil(len(active_classes) / cols))

for idx, cls in enumerate(active_classes):
    cls_folder = os.path.join(train_dir, cls)
    sample_file = random.choice([f for f in os.listdir(cls_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    img = Image.open(os.path.join(cls_folder, sample_file)).convert('RGB')
    
    plt.subplot(rows, cols, idx + 1)
    plt.imshow(img)
    plt.title(cls.replace('_', ' ').title(), fontsize=11)
    plt.axis("off")

plt.suptitle("Sample Real-World Images (Notice Natural Backgrounds & Context)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 5. Data Preprocessing & Augmentations for Real-World Robustness
To help our custom CNN learn robust representations from real-world photography:
1. **Random Resized Crop (128x128):** Zooms into different areas of the fruit and background, preventing the model from assuming the fruit is always centered.
2. **Random Horizontal Flip ($p=0.5$):** Enhances orientation invariance.
3. **Color Jitter (Brightness, Contrast, Saturation):** Simulates different indoor kitchen lights without distorting hue.
4. **Normalization:** ImageNet mean and standard deviation scaling.


In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 32

train_transforms = transforms.Compose([
    transforms.Resize((140, 140)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders ready:")
print(f"  Train: {len(train_dataset)} images across {len(train_loader)} batches")
print(f"  Test:  {len(test_dataset)} images across {len(test_loader)} batches")


## 6. Building our Custom Real-World CNN Architecture
Because real-world photos feature varied lighting and background clutter, we enhance our custom architecture to have **4 convolutional blocks** with **Batch Normalization** and **Dropout**:

```
Input Image (3 x 128 x 128)
    │
    ▼
[Conv2d (3 -> 32) + BatchNorm + ReLU] ──> MaxPool2d ──> (32 x 64 x 64)
    │
    ▼
[Conv2d (32 -> 64) + BatchNorm + ReLU] ──> MaxPool2d ──> (64 x 32 x 32)
    │
    ▼
[Conv2d (64 -> 128) + BatchNorm + ReLU] ──> MaxPool2d ──> (128 x 16 x 16)
    │
    ▼
[Conv2d (128 -> 256) + BatchNorm + ReLU] ──> AdaptiveAvgPool2d(4, 4) ──> (256 x 4 x 4 = 4096)
    │
    ▼
[Dropout(0.4)] ──> [Linear(4096 -> 512) + ReLU] ──> [Dropout(0.3)] ──> [Linear(512 -> num_classes)]
```


In [ ]:
class RealWorldFruitCNN(nn.Module):
    def __init__(self, num_classes=len(active_classes)):
        super(RealWorldFruitCNN, self).__init__()
        
        # Block 1: Basic edges & color gradients
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)  # 128 -> 64
        )
        
        # Block 2: Textures & shapes
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)  # 64 -> 32
        )
        
        # Block 3: Complex object contours
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)  # 32 -> 16
        )
        
        # Block 4: High-level semantic features
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)) # 256 x 4 x 4 = 4096
        )
        
        # Classifier Head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.4),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        logits = self.classifier(x)
        return logits

model = RealWorldFruitCNN(num_classes=len(active_classes)).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal Trainable Parameters: {total_params:,}")


## 7. Model Training & Optimization Loop
We train our custom model from scratch using `CrossEntropyLoss` and `Adam`.  
Checkpoints are automatically saved to your Google Drive (`Minor AI/models/best_realworld_cnn.pt`).


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

EPOCHS = 20
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
checkpoint_path = os.path.join(SAVE_DIR, 'best_realworld_cnn.pt')

print(f"Training Custom Real-World CNN for {EPOCHS} epochs on {device}...")

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
        
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = (correct_train / total_train) * 100.0
    
    # Validation
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100.0
    
    scheduler.step(epoch_val_acc)
    
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS}] | "
          f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.2f}%")
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'classes': active_classes
        }, checkpoint_path)
        print(f"  --> Saved new best checkpoint to {checkpoint_path} ({best_val_acc:.2f}%)")

print(f"\nTraining complete! Peak Validation Accuracy: {best_val_acc:.2f}%")


## 8. Quantitative Evaluation & Learning Curves
We plot the training and validation curves across all 20 epochs to observe model convergence on real-world imagery.


In [ ]:
plt.figure(figsize=(14, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), history['train_loss'], label='Train Loss', marker='o')
plt.plot(range(1, EPOCHS + 1), history['val_loss'], label='Val Loss', marker='s')
plt.title("Loss Curves (Real-World CNN)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), history['train_acc'], label='Train Acc', marker='o')
plt.plot(range(1, EPOCHS + 1), history['val_acc'], label='Val Acc', marker='s')
plt.title("Accuracy Curves (Real-World CNN)")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


## 9. Error Analysis: Confusion Matrix & Classification Report


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds, all_targets = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.numpy())

cm = confusion_matrix(all_targets, all_preds)
class_labels = [c.replace('_', ' ').title() for c in active_classes]

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels,
            yticklabels=class_labels)
plt.title("Confusion Matrix (Best Model Checkpoint)")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(all_targets, all_preds, target_names=class_labels, digits=3))


## 10. Qualitative Prediction Inspection
Visual testing on sample test images with confidence scores and color-coded tags.


In [ ]:
def denormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.cpu().numpy().transpose((1, 2, 0))
    img = std * img + mean
    return np.clip(img, 0, 1)

model.eval()
indices = random.sample(range(len(test_dataset)), min(10, len(test_dataset)))

plt.figure(figsize=(15, 6))
cols = 5
rows = int(np.ceil(len(indices) / cols))

with torch.no_grad():
    for plot_idx, idx in enumerate(indices):
        img_tensor, true_label = test_dataset[idx]
        output = model(img_tensor.unsqueeze(0).to(device))
        probs = torch.softmax(output, dim=1)
        conf, pred_label = torch.max(probs, 1)
        
        pred_cls = active_classes[pred_label.item()].replace('_', ' ').title()
        true_cls = active_classes[true_label].replace('_', ' ').title()
        is_correct = (pred_label.item() == true_label)
        
        plt.subplot(rows, cols, plot_idx + 1)
        plt.imshow(denormalize(img_tensor))
        color = 'green' if is_correct else 'red'
        plt.title(f"True: {true_cls}\nPred: {pred_cls} ({conf.item()*100:.1f}%)", color=color, fontsize=10)
        plt.axis("off")

plt.suptitle("Sample Predictions on Real-World Test Images", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 11. Interactive Smartphone Scanner App (Gradio)
Launch our custom-trained CNN in a real-time web interface. With `share=True`, you can scan real fruits and vegetables using your phone camera!


In [ ]:
!pip install -q gradio

import gradio as gr
import torch
from PIL import Image
from torchvision import transforms

# 1. Load the best saved model weights
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
classes = [c.replace('_', ' ').title() for c in checkpoint['classes']]

# 2. Inference preprocessing pipeline
inference_transforms = transforms.Compose([
    transforms.Resize((140, 140)),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 3. Prediction function
def scan_real_fruit(image):
    if image is None:
        return {}
    
    img = Image.fromarray(image).convert('RGB')
    tensor = inference_transforms(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        
    confidences = {classes[i]: float(probs[i]) for i in range(len(classes))}
    return confidences

# 4. Gradio UI
app = gr.Interface(
    fn=scan_real_fruit,
    inputs=gr.Image(label="Take a photo or upload an image", sources=['camera', 'upload']),
    outputs=gr.Label(num_top_classes=3, label="Top Predictions"),
    title="🍎 Real-World Fruit & Vegetable Scanner",
    description="Trained 100% from scratch on real-world photos! Scan real fruits/vegetables with your phone camera."
)

app.launch(share=True, debug=False)


## 12. Reflection & Next Steps
### What This Demonstrates:
1. **Solving Problems with Data Quality:** Instead of relying on pre-trained networks, we directly tackled the *domain gap* by training our custom CNN on real-world in-the-wild photography.
2. **From-Scratch Validation:** Proves that an appropriately regularized 4-block CNN can extract invariant features even with backgrounds and real lighting.
3. **Bridge to Food-101:** Having conquered isolated fruits and real vegetables from scratch, we are now ready for multi-ingredient complex meal plates and calorie regression in Notebooks 2 & 3.
